# Clean reference backtest
The universe is fixed in advance, signals are shifted, and normalization expands only from past data.

In [ ]:
import numpy as np
import pandas as pd

EX_ANTE_UNIVERSE = tuple("ABCD")

def signal(prices):
    """Use lagged returns and expanding statistics available before each bar."""
    fixed_prices = prices.loc[:, list(EX_ANTE_UNIVERSE)]
    past_returns = fixed_prices.pct_change().shift(1)
    mean = past_returns.expanding(min_periods=2).mean()
    std = past_returns.expanding(min_periods=2).std().replace(0.0, np.nan)
    return ((past_returns - mean) / std).fillna(0.0)

def backtest(prices):
    positions = signal(prices)
    asset_returns = prices.loc[:, list(EX_ANTE_UNIVERSE)].pct_change().fillna(0.0)
    return (positions * asset_returns).mean(axis=1)

def synthetic_prices():
    rng = np.random.default_rng(42)
    returns = rng.normal(0.001, 0.01, size=(120, 4))
    return pd.DataFrame(100.0 * (1.0 + returns).cumprod(axis=0), columns=list("ABCD"))
